# Porównawcza wersja sieci napisana w bibliotece torch

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import torch.utils.benchmark as benchmark
from torchvision import datasets, transforms
import tracemalloc

In [2]:
class CNNNetwork(nn.Module):
    def __init__(self, dropout_p=0.4):
        super(CNNNetwork, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=3, stride=1, padding=0)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=3, stride=1, padding=0)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 5 * 5, 84)  # After two 3x3 convs and two 2x2 pools on 28x28
        self.dropout = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(84, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [3]:
lr = 0.01
batch_size = 1
dropout_p = 0.4
bench_samples = 60

In [4]:
def train_bench(model, x, y, optimizer, batch_size):
    model.train()
    N = x.size(0)
    for batch_start in range(0, N, batch_size):
        batch_end = min(batch_start + batch_size, N)
        batch_x = x[batch_start:batch_end]
        batch_y = y[batch_start:batch_end]
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

In [7]:
transform = transforms.ToTensor()
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)

x_mini = []
y_mini = []
for i in range(bench_samples):
    x, y = train_data[i]
    x_mini.append(x)
    y_mini.append(y)

x_mini = torch.stack(x_mini)
y_mini = torch.tensor(y_mini)

In [8]:
model = CNNNetwork(dropout_p=dropout_p)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr)

In [9]:
timer = benchmark.Timer(
    stmt='train_bench(model, x_mini, y_mini, optimizer, batch_size)',
    globals={'train_bench': train_bench, 'model': model, 'x_mini': x_mini, 
             'y_mini': y_mini, 'optimizer': optimizer, 'batch_size': batch_size}
)
result = timer.timeit(number=10)
print(result)

train_bench(model, x_mini, y_mini, optimizer, batch_size)
  36.81 ms
  1 measurement, 10 runs , 1 thread


In [ ]:
tracemalloc.start()
for _ in range(50):
    train_bench(model, x_mini, y_mini, optimizer, batch_size)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Peak memory usage: {peak / 1024 / 1024:.2f} MB")

Peak memory usage: 0.02 MB


In [12]:
import psutil
import os

process = psutil.Process(os.getpid())

# Get memory before
mem_before = process.memory_info().rss / 1024 / 1024

for _ in range(50):
    train_bench(model, x_mini, y_mini, optimizer, batch_size)

mem_after = process.memory_info().rss / 1024 / 1024
print(f"Memory used: {mem_after - mem_before:.2f} MB")
print(f"Total process memory: {mem_after:.2f} MB")

Memory used: 0.05 MB
Total process memory: 395.32 MB
